# 04. Audit Sinks — Tamper-evident logging

Every security-relevant action in Arc emits an `AuditEvent`. The model called a tool. The policy pipeline returned DENY. A child identity was derived. A signature failed to verify. Each of these is a single immutable event with the same shape, written to one or more pluggable sinks at the moment it happens.

This notebook walks through the audit module of `arctrust` — the canonical implementation of the **Audit** pillar from ADR-019. It is mock-first, runs without an API key, and writes everything into `tmp_path`-style temp directories.

**You will learn:**

- Why audit is a universal Arc primitive, not a federal-mode feature
- How `AuditEvent` is shaped — required fields, optional metadata, frozen-ness
- How `emit()` dispatches to a sink and *never* lets a sink failure break the caller
- How `JsonlSink` writes append-only NIST-grade compliance logs
- How `SignedChainSink` builds an Ed25519 hash chain that detects tampering
- How to write a custom `AuditSink` (it's just one method)
- How to fan a single `emit()` out to multiple sinks at once


## 1. Setup

Run the boilerplate below once. It walks up to the Arc repo root and adds every `packages/<pkg>/src` and `packages/<pkg>/tests` directory to `sys.path`. After this cell, `import arctrust` works directly from the source tree.

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Confirm the audit surface is importable. We'll lean on every name listed below as the notebook progresses.

In [ ]:
from arctrust import AuditEvent, AuditSink, JsonlSink, NullSink, SignedChainSink, emit
from arctrust import generate_keypair

print("AuditEvent       :", AuditEvent)
print("AuditSink        :", AuditSink)
print("JsonlSink        :", JsonlSink)
print("SignedChainSink  :", SignedChainSink)
print("NullSink         :", NullSink)
print("emit             :", emit)

## 2. Why audit — the federal compliance lens

Audit is one of the **four pillars** every Arc deployment enforces (ADR-019):

1. **Identity** — every entity has a DID (covered in `01-identity-did`)
2. **Sign** — every loaded artifact is verified before use (`02-keypairs-signing`)
3. **Authorize** — every tool call passes through a policy pipeline (`03-policy-pipeline`)
4. **Audit** — every operation is recorded as an immutable event (this notebook)

Tier (personal / enterprise / federal) is *stringency metadata*, not a gate. Federal demands FIPS-validated crypto and signed allowlists; personal allows a self-signed bundle. **Every tier still audits.**

The compliance hooks are concrete:

| NIST 800-53 control | What it requires | Where arctrust covers it |
|---|---|---|
| **AU-2** Event types | Capture security-relevant events with actor, action, outcome | `AuditEvent` schema |
| **AU-5** Failure response | Audit failure must not interrupt the audited operation | `emit()` swallows sink errors |
| **AU-9** Protection | Detect tampering of stored audit records | `SignedChainSink` hash chain |
| **AU-11** Retention | Append-only, immutable storage | `JsonlSink` open-mode `"a"` |

FedRAMP and CMMC inherit these AU controls. The same code base runs in a developer's home directory and on a DOE machine in a SCIF — only the sinks and the operator key change.

## 3. The `AuditEvent` schema — what gets recorded

An `AuditEvent` is a frozen Pydantic model. Once constructed, it cannot be mutated — that's the whole point. Audit data must be a record of what happened, not a buffer that something later edits.

Required fields capture the universal who/what/when/outcome:

In [ ]:
evt = AuditEvent(
    actor_did="did:arc:walkthrough:exec/aabbccdd",
    action="tool.call",
    target="read_file",
    outcome="allow",
)

print("actor_did :", evt.actor_did)
print("action    :", evt.action)
print("target    :", evt.target)
print("outcome   :", evt.outcome)
print("ts        :", evt.ts)

Notice the `ts` field auto-populates with an ISO 8601 UTC timestamp at construction time. We never trust the caller to pass a timestamp — the audit module owns that.

Optional fields cover request correlation, tier metadata, classification, and tamper-evidence of the request payload itself (a SHA-256, never raw data):

In [ ]:
import hashlib

payload = b'{"path": "/etc/passwd"}'
evt_full = AuditEvent(
    actor_did="did:arc:walkthrough:exec/aabbccdd",
    action="tool.call",
    target="read_file",
    outcome="deny",
    classification="UNCLASSIFIED",
    tier="federal",
    request_id="req-2026-05-07-0001",
    payload_hash=hashlib.sha256(payload).hexdigest(),
    extra={"reason": "path not on allowlist"},
)

for k, v in evt_full.model_dump().items():
    print(f"  {k:<14}: {v}")

The frozen-ness is not advisory. Pydantic enforces it — any attempt to mutate a field on an event after creation raises:

In [ ]:
try:
    evt.outcome = "allow"  # type: ignore[misc]
except Exception as e:
    print("raised:", type(e).__name__, str(e).splitlines()[0])

Because the model is fully serialisable, every event round-trips through JSON without losing information — the foundation for every sink we'll see next.

In [ ]:
import json

as_json = json.dumps(evt_full.model_dump(), default=str, indent=2)
print(as_json)

## 4. The `emit()` function — single point of dispatch

Every part of Arc that needs to audit something does the same thing: build an `AuditEvent`, then call `emit(event, sink)`. That's it. The emit function is *deliberately* tiny:

```python
def emit(event: AuditEvent, sink: AuditSink) -> None:
    try:
        sink.write(event)
    except Exception:
        _logger.warning(...)  # AU-5: never propagate
```

The contract is: **the audit subsystem will never break the operation it is auditing**. If the disk is full, if the network sink is down, if a custom sink throws — `emit` swallows it, logs at WARNING, and returns. That's NIST AU-5 (Response to Audit Processing Failures).

Let's see this in action with a deliberately broken sink:

In [ ]:
class ExplodingSink:
    """A sink that raises on every write — to demonstrate AU-5 behaviour."""

    def write(self, event: AuditEvent) -> None:
        raise RuntimeError("disk on fire")


evt = AuditEvent(
    actor_did="did:arc:walkthrough:exec/aabbccdd",
    action="emit.demo",
    target="broken-sink",
    outcome="allow",
)

# emit() must NOT raise. The caller's path is preserved.
emit(evt, ExplodingSink())  # type: ignore[arg-type]
print("caller still alive — emit() ate the RuntimeError, exactly per AU-5")

If we had not used `emit()` and called `sink.write()` directly, the runtime would have crashed and the audited operation would have failed. That's why arctrust never recommends calling sinks directly — `emit()` is the safe API.

## 5. `JsonlSink` — append-only compliance log

The most common sink writes one JSON object per line into a `.jsonl` file. The file is opened in append mode (`"a"`) — restarts, multiple processes, multiple sink instances all accumulate into the same file in order. That's NIST AU-11 (Audit Record Retention) at the file level.

We'll use `tmp_path` semantics — a `tempfile.TemporaryDirectory` so the demo is hermetic:

In [ ]:
import tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp(prefix="arctrust-audit-"))
log_path = tmp / "audit.jsonl"
print("log_path:", log_path)

Construct a `JsonlSink` pointing at the path and emit a few events through it:

In [ ]:
jsonl = JsonlSink(log_path)

for i in range(4):
    evt = AuditEvent(
        actor_did=f"did:arc:walkthrough:exec/{i:08x}",
        action="tool.call",
        target="read_file",
        outcome="allow" if i % 2 == 0 else "deny",
        request_id=f"req-{i:04d}",
    )
    emit(evt, jsonl)

print("file size:", log_path.stat().st_size, "bytes")
print("line count:", len(log_path.read_text().splitlines()))

Read records back as plain JSON lines — this is the format compliance scrapers expect:

In [ ]:
import json

for raw in log_path.read_text().splitlines():
    rec = json.loads(raw)
    print(
        f"  {rec['ts']}  {rec['action']:<10}  {rec['target']:<10}  {rec['outcome']}  ({rec['actor_did']})"
    )

Two important properties of `JsonlSink` worth calling out explicitly:

1. **Append-only at the OS level.** The constructor doesn't truncate; each `write()` opens    in `"a"`. Restarts are safe.
2. **File-like accepted.** You can pass a `StringIO` for tests, or wrap a syslog stream.    The sink doesn't care.

In [ ]:
from io import StringIO

buf = StringIO()
buf_sink = JsonlSink(buf)  # type: ignore[arg-type]

evt = AuditEvent(
    actor_did="did:arc:walkthrough:exec/aabbccdd",
    action="tool.call",
    target="in-memory",
    outcome="allow",
)
emit(evt, buf_sink)
print("in-memory line:", buf.getvalue().strip())

## 6. `SignedChainSink` — Ed25519 hash chain for tamper evidence

A `JsonlSink` is honest about what it isn't: a passive recorder. If an attacker with disk access edits a single character of a stored record, nothing in `JsonlSink` notices.

`SignedChainSink` raises that bar. Every record carries:

- The full event payload
- The hash of the *previous* record (`prev_hash`) — the chain link
- The SHA-256 of `(prev_hash + canonical_event_json)` (`event_hash`)
- An Ed25519 signature over `event_hash` using the operator's private key

Hash chaining means: tamper with record N and record N+1's `prev_hash` no longer matches. Re-hash the entire suffix and the signatures still don't match the operator key. This is the concrete implementation of NIST AU-9 (Protection of Audit Information).

Generate an operator keypair and write a few signed events:

In [ ]:
operator = generate_keypair()
chain = SignedChainSink(operator_private_key=operator.private_key)

print("chain tip before any writes:", repr(chain.chain_tip))

for i in range(4):
    evt = AuditEvent(
        actor_did=f"did:arc:walkthrough:exec/{i:08x}",
        action="policy.evaluate",
        target=f"tool_{i}",
        outcome="allow" if i % 2 == 0 else "deny",
    )
    emit(evt, chain)

print("chain tip after 4 writes:", chain.chain_tip[:24], "…")
print("records stored         :", len(chain.records))

Inspect the structure of one signed record. The combination of `prev_hash`, `event_hash`, and `signature` is what makes the chain tamper-evident:

In [ ]:
import json

rec = chain.records[2]
print("record keys:", list(rec.keys()))
print("prev_hash  :", rec["prev_hash"][:24], "…")
print("event_hash :", rec["event_hash"][:24], "…")
print(
    "signature  :",
    rec["signature"][:24],
    "…",
    "(" + str(len(bytes.fromhex(rec["signature"]))) + " bytes)",
)
print("event keys :", list(rec["event"].keys()))

An intact chain verifies cleanly. `verify_chain()` walks every record from genesis, recomputing the expected hash and confirming it matches what was stored:

In [ ]:
ok = chain.verify_chain()
print("intact chain verifies:", ok)
assert ok, "freshly written chain must verify"

## 7. Tampering demo — what a chain breach looks like

Now we play attacker. Modify a stored record's event content directly in memory and observe that `verify_chain()` returns `False`. In a real deployment the records would live on disk; the verification logic is the same.

First, capture the current state and pick a record to mutate:

In [ ]:
target_index = 1
original = chain.records[target_index]
print("target record action (before):", original["event"]["action"])
print("target record outcome (before):", original["event"]["outcome"])

Mutate the event content in place. Imagine an attacker rewriting a `deny` to an `allow` to hide the fact that they were rejected by policy:

In [ ]:
tampered_event = {**original["event"], "outcome": "allow", "action": "tampered_action"}
tampered_record = {**original, "event": tampered_event}
chain.records[target_index] = tampered_record

print("target record action (after) :", chain.records[target_index]["event"]["action"])
print("target record outcome (after):", chain.records[target_index]["event"]["outcome"])

Run verification — and watch the breach surface. We wrap the call in a small helper that reports the failure clearly, the way an audit-monitoring tool would:

In [ ]:
def verify_or_alert(sink: SignedChainSink, label: str) -> None:
    try:
        ok = sink.verify_chain()
        if not ok:
            print(f"tamper detected: chain integrity check failed for {label!r}")
        else:
            print(f"chain intact for {label!r}")
    except Exception as exc:
        print(f"tamper detected: {label!r} raised {type(exc).__name__}: {exc}")


verify_or_alert(chain, "after-tamper")

The chain has been broken. The first hash mismatch means every downstream record's `prev_hash` lineage no longer fits — even if the attacker re-signed only the tampered record, the signature was over the original `event_hash`, not the new one. To produce a valid chain the attacker would need the operator's private key (which is precisely the point of holding it out-of-band).

This is what tamper-evident logging means in practice: the moment somebody edits the audit trail, your monitoring sees it. Even if they delete the breach detection logs themselves, the chain tip stored in a separate (write-only, signed) place tells you the chain has been shortened.

Restore the record and confirm verification passes again. (In real systems you wouldn't restore — you'd alert and quarantine the log file; we restore here so the rest of the notebook continues from a clean state.)

In [ ]:
chain.records[target_index] = original
verify_or_alert(chain, "restored")

## 8. Writing a custom `AuditSink`

`AuditSink` is a runtime-checkable Protocol — *any* object with a `write(event)` method satisfies it. No inheritance, no mixin, no registration step. This is deliberate: every deployment has its own infrastructure (Splunk, Elastic, CloudWatch, syslog, NATS, plain stdout, an in-memory buffer for tests) and the audit module shouldn't care which.

Here are three custom sinks in under 30 lines combined:

In [ ]:
class InMemorySink:
    """Capture events into a list. Useful for tests and assertions."""

    def __init__(self) -> None:
        self.events: list[AuditEvent] = []

    def write(self, event: AuditEvent) -> None:
        self.events.append(event)


class CountingSink:
    """Count events by (action, outcome). The shape of a metrics emitter."""

    def __init__(self) -> None:
        self.counts: dict[tuple[str, str], int] = {}

    def write(self, event: AuditEvent) -> None:
        key = (event.action, event.outcome)
        self.counts[key] = self.counts.get(key, 0) + 1


class StdoutSink:
    """Print a one-line human-readable summary. Useful in dev."""

    def write(self, event: AuditEvent) -> None:
        print(f"[{event.ts}] {event.actor_did} {event.action}({event.target}) -> {event.outcome}")

All three pass the `isinstance(sink, AuditSink)` Protocol check at runtime — no inheritance required:

In [ ]:
in_mem = InMemorySink()
counts = CountingSink()
stdout = StdoutSink()

for sink in (in_mem, counts, stdout):
    print(f"{type(sink).__name__:<14} isinstance(AuditSink) -> {isinstance(sink, AuditSink)}")

Drive each sink through `emit()` and confirm they behave as expected:

In [ ]:
for action, outcome in [
    ("tool.call", "allow"),
    ("tool.call", "deny"),
    ("policy.evaluate", "allow"),
    ("policy.evaluate", "deny"),
    ("tool.call", "allow"),
]:
    evt = AuditEvent(
        actor_did="did:arc:walkthrough:exec/aabbccdd",
        action=action,
        target="some_tool",
        outcome=outcome,
    )
    emit(evt, in_mem)
    emit(evt, counts)

print("InMemorySink   ->", len(in_mem.events), "events captured")
print("CountingSink   ->")
for (action, outcome), n in counts.counts.items():
    print(f"    ({action:<18}, {outcome:<5}): {n}")

`NullSink` ships in `arctrust` for the special case where you genuinely want to discard every event — typically test fixtures or air-gapped evaluation harnesses where you don't want side effects. It satisfies the Protocol with a no-op `write`:

In [ ]:
null = NullSink()
evt = AuditEvent(
    actor_did="did:arc:walkthrough:exec/aabbccdd",
    action="tool.call",
    target="black_hole",
    outcome="allow",
)
emit(evt, null)
print("NullSink records (always empty):", null.records)

## 9. Multi-sink fan-out — the production pattern

Real Arc deployments rarely use a single sink. The typical wiring fans every event out to:

- A **`JsonlSink`** for compliance archive (cheap, append-only, easy to ship)
- A **`SignedChainSink`** for tamper evidence (Ed25519, AU-9)
- A **metrics sink** for live observability (counts by action/outcome)
- An **arcui bridge sink** so dashboards see events the moment they fire

There's no `MultiSink` class because there doesn't need to be. A four-line tuple plus a loop is the entire pattern, and `emit()` already handles per-sink failures:

In [ ]:
import tempfile
from pathlib import Path

fanout_dir = Path(tempfile.mkdtemp(prefix="arctrust-fanout-"))
compliance = JsonlSink(fanout_dir / "compliance.jsonl")
tamper_chain = SignedChainSink(operator_private_key=operator.private_key)
metrics = CountingSink()
memory = InMemorySink()

all_sinks: tuple[AuditSink, ...] = (compliance, tamper_chain, metrics, memory)


def fan_out(event: AuditEvent, sinks: tuple[AuditSink, ...]) -> None:
    for s in sinks:
        emit(event, s)  # emit() isolates per-sink failures

Drive a small workflow's worth of audit events through the fan-out — the kind of trace a single agent run would produce:

In [ ]:
workflow_events = [
    ("agent.start", "agent_loop", "allow"),
    ("policy.evaluate", "read_file", "allow"),
    ("tool.call", "read_file", "allow"),
    ("policy.evaluate", "write_file", "deny"),
    ("tool.call", "write_file", "deny"),
    ("agent.complete", "agent_loop", "allow"),
]

for action, target, outcome in workflow_events:
    evt = AuditEvent(
        actor_did="did:arc:walkthrough:exec/agent01",
        action=action,
        target=target,
        outcome=outcome,
        tier="federal",
        request_id="workflow-001",
    )
    fan_out(evt, all_sinks)

print("compliance file lines:", len((fanout_dir / "compliance.jsonl").read_text().splitlines()))
print("signed chain records :", len(tamper_chain.records))
print("metrics counts       :")
for k, v in metrics.counts.items():
    print(f"    {k}: {v}")
print("memory events        :", len(memory.events))

Every sink saw every event. The compliance file is on disk, the signed chain verifies, the metrics counter is up to date, and an in-memory copy is available for tests or live observability. One `fan_out()` call, four storage backends, zero coupling.

And critically — if any one sink had failed (full disk, network blip), `emit()` would have isolated the failure to that sink and the others would still have received the event. Let's prove it by mixing a working sink with the deliberately broken one:

In [ ]:
resilient_sinks: tuple[AuditSink, ...] = (compliance, ExplodingSink(), memory)  # type: ignore[arg-type]

evt = AuditEvent(
    actor_did="did:arc:walkthrough:exec/agent01",
    action="resilience.test",
    target="mixed-sinks",
    outcome="allow",
)
fan_out(evt, resilient_sinks)

print("compliance line count :", len((fanout_dir / "compliance.jsonl").read_text().splitlines()))
print("memory event count    :", len(memory.events))
print("caller still alive    : True")

The exploding sink raised, the warning was logged, the other two sinks still received the event, and the calling code carried on. That's AU-5 fan-out resilience in 30 lines.

## 10. Summary

**API surface covered:**

| Symbol | Role |
|---|---|
| `AuditEvent` | Frozen Pydantic schema for all security-relevant events |
| `AuditSink` | Runtime-checkable Protocol — any `write(event)` method qualifies |
| `JsonlSink` | Append-only JSONL file sink (NIST AU-11) |
| `SignedChainSink` | Ed25519 hash-chained tamper-evident sink (NIST AU-9) |
| `NullSink` | No-op sink for tests and air-gapped evaluation |
| `emit()` | Safe dispatch — swallows sink failures (NIST AU-5) |

**Patterns demonstrated:**

- Build an event, hand it to `emit(event, sink)` — never call sinks directly
- Mutating a stored signed record breaks `verify_chain()`
- Custom sinks are a one-method protocol implementation; no inheritance, no registration
- Fan-out is a tuple plus a loop — there's no `MultiSink` because there doesn't need to be
- A failing sink in a fan-out does not block the other sinks or the caller

**Compliance hooks landed:**

- AU-2 (event capture) via `AuditEvent`
- AU-5 (failure response) via `emit()`
- AU-9 (protection of audit information) via `SignedChainSink`
- AU-11 (retention) via `JsonlSink` append-mode

**Where the audit story continues:**

- `walkthroughs/arcllm/10-audit-trail.ipynb` — `AuditModule` in the arcllm module stack,   emitting through the same `arctrust.audit.emit` we just used.
- `walkthroughs/arcllm/14-trace-store.ipynb` — Hash-chained `TraceStore` for full LLM call   traces with daily rotation and warm-start tamper detection.

Both build directly on the primitives in this notebook. The audit pillar starts here.